# Öz LLM-im: sıfırdan kiçik GPT (Azərbaycan dili)

Bu notebook Azərbaycan Vikipediyası üzərində **sıfırdan** kiçik GPT modeli öyrədir.

**Kaggle-da işə salmazdan əvvəl** (sağ paneldə, *Session options*):
1. **Accelerator** → `GPU T4 x2` və ya `GPU P100`
2. **Internet** → `On` (bunun üçün hesabda telefon nömrəsi təsdiqlənməlidir)

Sonra yuxarıdan **Run All** et. Sonda `/kaggle/working/my_llm.zip` faylı yaranacaq. Onu sağdakı *Output* bölməsindən endir.

In [ ]:
# 1) Ayarlar: istəsən bunları dəyiş
MAX_CHARS   = 300_000_000  # nə qədər mətn götürək (simvol). Az RAM varsa azalt
VOCAB_SIZE  = 8000         # tokenizer lüğətinin ölçüsü
BLOCK_SIZE  = 256          # kontekst uzunluğu (token)
N_LAYER, N_HEAD, N_EMBD = 6, 6, 384   # ~14M parametr. Daha böyük: 8, 8, 512
BATCH_SIZE  = 32
LR          = 6e-4
MAX_STEPS   = 15_000
MAX_MINUTES = 150          # bu qədər dəqiqədən sonra təlim özü dayanır
EVAL_EVERY  = 500
SAMPLE_PROMPT = "Azərbaycan"

!pip install -q datasets tokenizers

In [ ]:
%%writefile model.py
"""Kiçik GPT modeli (nanoGPT üslubunda). Həm Kaggle-da təlim, həm də lokal sayt bunu istifadə edir."""
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


@dataclass
class GPTConfig:
    vocab_size: int = 8000   # tokenizerdəki "söz parçaları"nın sayı
    block_size: int = 256    # model bir dəfəyə neçə tokenə baxır (kontekst uzunluğu)
    n_layer: int = 6         # transformer blok sayı
    n_head: int = 6          # attention "baş"larının sayı
    n_embd: int = 384        # hər tokenin vektor ölçüsü
    dropout: float = 0.1


class Block(nn.Module):
    """Bir transformer bloku: əvvəl attention (tokenlər bir-birinə baxır), sonra MLP (hər token özü düşünür)."""

    def __init__(self, c: GPTConfig):
        super().__init__()
        self.n_head = c.n_head
        self.dropout = c.dropout
        self.ln1 = nn.LayerNorm(c.n_embd)
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.proj = nn.Linear(c.n_embd, c.n_embd)
        self.ln2 = nn.LayerNorm(c.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(c.n_embd, 4 * c.n_embd),
            nn.GELU(),
            nn.Linear(4 * c.n_embd, c.n_embd),
            nn.Dropout(c.dropout),
        )
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # is_causal=True: hər token yalnız özündən əvvəlkilərə baxa bilir (gələcəyi görmür)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        x = x + self.drop(self.proj(y))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, c: GPTConfig):
        super().__init__()
        self.config = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.n_embd)
        self.pos_emb = nn.Embedding(c.block_size, c.n_embd)
        self.drop = nn.Dropout(c.dropout)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # giriş və çıxış çəkiləri ortaqdır (parametrə qənaət)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=0.8, top_k=40, stop_id=None):
        """Hər addımda növbəti tokeni ehtimala görə seçib sona əlavə edir."""
        for _ in range(max_new_tokens):
            ctx = idx[:, -self.config.block_size:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            if stop_id is not None and nxt.item() == stop_id:
                break
            idx = torch.cat([idx, nxt], dim=1)
        return idx


In [ ]:
# 2) Data: Azərbaycan Vikipediyası
from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.az", split="train")
texts, total = [], 0
for t in ds["text"]:
    t = t.strip()
    if len(t) < 200:          # çox qısa (boş) məqalələri at
        continue
    texts.append(t)
    total += len(t)
    if total >= MAX_CHARS:
        break
print(f"{len(texts):,} məqalə, {total/1e6:.0f} milyon simvol")
print(texts[0][:300])

In [ ]:
# 3) Tokenizer: mətni "söz parçaları"na bölməyi öyrənir
from tokenizers import ByteLevelBPETokenizer

tok = ByteLevelBPETokenizer()
tok.train_from_iterator(texts, vocab_size=VOCAB_SIZE, min_frequency=2,
                        special_tokens=["<|endoftext|>"])
tok.save("tokenizer.json")
EOT = tok.token_to_id("<|endoftext|>")
print("Nümunə:", tok.encode("Bakı Azərbaycanın paytaxtıdır.").tokens)

In [ ]:
# 4) Bütün mətni rəqəmlərə (token id) çevir
import numpy as np

parts = []
for i in range(0, len(texts), 2000):
    for e in tok.encode_batch(texts[i:i+2000]):
        parts.append(np.array(e.ids + [EOT], dtype=np.uint16))
data = np.concatenate(parts); del parts
split = int(len(data) * 0.95)
train_data, val_data = data[:split], data[split:]
print(f"Cəmi {len(data)/1e6:.1f} milyon token (train {len(train_data)/1e6:.1f}M, val {len(val_data)/1e6:.1f}M)")

In [ ]:
# 5) Təlim
import math, time, torch
from dataclasses import asdict
from model import GPT, GPTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
print("Cihaz:", device)

cfg = GPTConfig(vocab_size=tok.get_vocab_size(), block_size=BLOCK_SIZE,
                n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD)
model = GPT(cfg).to(device)
print(f"Model: {model.num_params()/1e6:.1f} milyon parametr")

opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.1)
scaler = torch.amp.GradScaler(enabled=use_amp)

def get_batch(d):
    ix = np.random.randint(0, len(d) - BLOCK_SIZE - 1, BATCH_SIZE)
    x = torch.from_numpy(np.stack([d[i:i+BLOCK_SIZE] for i in ix]).astype(np.int64))
    y = torch.from_numpy(np.stack([d[i+1:i+1+BLOCK_SIZE] for i in ix]).astype(np.int64))
    return x.to(device), y.to(device)

def lr_at(step, warmup=300):
    if step < warmup:
        return LR * (step + 1) / warmup
    p = (step - warmup) / max(1, MAX_STEPS - warmup)
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(p, 1.0))))

@torch.no_grad()
def eval_loss(d, n=40):
    model.eval(); s = 0.0
    for _ in range(n):
        x, y = get_batch(d)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            s += model(x, y)[1].item()
    model.train(); return s / n

def sample(prompt, n=60):
    model.eval()
    idx = torch.tensor([tok.encode(prompt).ids], device=device)
    out = model.generate(idx, max_new_tokens=n, stop_id=EOT)[0].tolist()
    model.train(); return tok.decode(out)

best, t0 = float("inf"), time.time()
for step in range(MAX_STEPS + 1):
    for g in opt.param_groups:
        g["lr"] = lr_at(step)
    x, y = get_batch(train_data)
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()

    mins = (time.time() - t0) / 60
    if step % EVAL_EVERY == 0 or step == MAX_STEPS or mins > MAX_MINUTES:
        vl = eval_loss(val_data)
        print(f"addım {step:>6} | train {loss.item():.3f} | val {vl:.3f} | {mins:.1f} dəq")
        if vl < best:
            best = vl
            torch.save({"config": asdict(cfg), "model": model.state_dict()}, "model.pt")
        print("   →", sample(SAMPLE_PROMPT).replace("\n", " ")[:200])
    if mins > MAX_MINUTES:
        print("Vaxt limiti doldu, təlim dayandırıldı."); break
print("Ən yaxşı val loss:", round(best, 3))

In [ ]:
# 6) Ən yaxşı modeli yüklə və sına
ck = torch.load("model.pt", map_location=device, weights_only=True)
model = GPT(GPTConfig(**ck["config"])).to(device); model.load_state_dict(ck["model"])
for p in ["Bakı şəhəri", "Azərbaycan Respublikası", "Nizami Gəncəvi"]:
    print(sample(p, 80), "\n---")

In [ ]:
# 7) Lokal sayt üçün lazım olan faylları zip-lə
import zipfile, os
out = "/kaggle/working/my_llm.zip" if os.path.isdir("/kaggle/working") else "my_llm.zip"
with zipfile.ZipFile(out, "w") as z:
    for f in ["model.pt", "tokenizer.json", "model.py"]:
        z.write(f)
print("Hazırdır:", out, f"({os.path.getsize(out)/1e6:.0f} MB)")